In [1]:
# Ejercicio 1:

In [2]:
def Redundante(L):
    # Número de restricciones
    m = len(L)
    # Variables del problema
    x, y = var('x,y')
    # Lista de restricciones que no son redundantes
    Validos = []

    # Comprobar las intersecciones de todas las rectas
    for i in [0..m-2]:
        for j in [i+1..m-1]:
            # Ecuaciones de las rectas
            eq1 = L[i][0]*x + L[i][1]*y == L[i][2]
            eq2 = L[j][0]*x + L[j][1]*y == L[j][2]
            # Calcular su intersección
            S = solve([eq1, eq2], x, y)

            # Si existe intersección:
            if len(S) > 0:
                # Coordenadas del punto
                x0 = S[0][0].rhs()
                y0 = S[0][1].rhs()

                # Comprobar si el vvertice cumple las restricciones
                factible = True
                for k in [0..m-1]:
                    if L[k][0]*x0 + L[k][1]*y0 > L[k][2]:
                        factible = False

                # Si es factible, las rectas que forman el vertice no son redudantes
                if factible:
                    if L[i] not in Validos:
                        Validos += [L[i]]
                    if L[j] not in Validos:
                        Validos += [L[j]]
    return Validos

In [3]:
def Resol1(P):
    # Número total de elementos
    m_total = len(P) 
    # Restricciones del problema
    L = [P[i] for i in [0..m_total-2]]
    # Función objetivo
    C = P[m_total-1] 
    # Eliminar restricciones redundantes
    Lval = Redundante(L)
    # Número de restricciones no redundantes
    m = len(Lval)
    # Variables del problema
    x, y = var('x,y')
    # Valor máximo inicial
    MaxZ = -infinity
    # Vértice óptimo inicial
    VerticeOptimo = []

    # Recorrer las restricciones no redundantes
    for i in [0..m-2]:
        for j in [i+1..m-1]:
            # Ecuaciones de las rectas
            eq1 = Lval[i][0]*x + Lval[i][1]*y == Lval[i][2]
            eq2 = Lval[j][0]*x + Lval[j][1]*y == Lval[j][2]
            # Calcular la intersección
            S = solve([eq1, eq2], x, y)

            # Si hay intersección:
            if len(S) > 0:
                # Coordenadas del punto
                x0 = S[0][0].rhs()
                y0 = S[0][1].rhs()
                
                # Comprobar si el vértice es factible
                factible = True
                for k in [0..m-1]:
                    if Lval[k][0]*x0 + Lval[k][1]*y0 > Lval[k][2]:
                        factible = False
                # Evaluar si mejora la funcion objetivo
                if factible:
                    Z = C[0]*x0 + C[1]*y0
                    if Z > MaxZ:
                        MaxZ = Z
                        VerticeOptimo = [x0, y0]
    return [VerticeOptimo, MaxZ]

In [4]:
# Comprobar
P1 = [[1,1,2], [1,-1,2], [3,2,9], [-1,1,2], [0,-1,3], [-1,-1,2], [2,3]]
print("Sin redundantes:", Redundante(P1[:-1]))
print("Solución:", Resol1(P1))

Sin redundantes: [[1, 1, 2], [1, -1, 2], [-1, 1, 2], [-1, -1, 2]]
Solución: [[0, 2], 6]


In [5]:
####################################

In [6]:
# Ejercicio 2:

In [7]:
def Col(A):
    # Número de filas sin contar la objetivo
    m = len(A) - 1
    # Número de columnas sin contar la de términos independientes
    n = len(A[0]) - 1
    # Lista donde se guardarán las columnas
    Col = []
    # Recorrer cada columna de variables
    for j in [0..n-1]:
        ColVar = []
        # Tomas los valores de la columna en las restricciones
        for i in [1..m]:
            ColVar += [A[i][j]]
        # Guardar la columna como matriz columna
        Col += [column_matrix(ColVar)]
    return Col

In [8]:
def Indep(A):
    # Número de filas sin contar la función objetivo
    m = len(A) -1
    # Índice de la columna de términos independientes
    n = len(A[0]) - 1
    # Lista para la columna de términos independientes
    Col = []
    # Coger la última columna pero sin la primera fila
    for i in [1..m]:
        Col += [A[i][n]]
    return column_matrix(Col)

In [9]:
def EntrSal(A, Entrada, Salida):
    # Número de filas sin contar la función objetivo
    m = len(A) - 1
    # Número de columnas sin contar términos independientes
    n = len(A[0]) - 1
    At = [[A[i][k] for k in [0..n]] for i in [0..m]]
    # Elemento pivote
    pivote = At[Salida][Entrada]
    # Dividir la fila pivote por el pivote
    for i in [0..n]:
        At[Salida][i] = At[Salida][i] / pivote
    # Hacer ceros en el resto de la columna de entrada
    for j in [0..m]:
        if j != Salida:
            aux = At[j][Entrada]
            for i in [0..n]:
                At[j][i] -= aux * At[Salida][i]
    return At

In [10]:
def FuncObj(A):
    # Número de columnas sin contar términos independientes
    n = len(A[0]) - 1
    # Comprobar si quedan coeficientes negativos
    for j in [0..n-1]:
        if A[0][j] < 0:
            return False
    # Si no hay negativosm se ha alcanzado el óptimo
    return True

In [11]:
def EncNoBasics(A):
    # Número de filas sin contar la función objetivo
    m = len(A) - 1
    # Número de columnas sin contar términos independientes
    n = len(A[0]) - 1
    # Lista para variables no básicas
    NoBasics = []
    # Recorrer todas las columnas
    for j in [0..n-1]:
        ceros = 0
        unos = 0
        # Contar cuántos ceros y unos tiene la columna
        for i in [1..m]:
            if A[i][j] == 0:
                ceros += 1
            if A[i][j] == 1:
                unos += 1
        # Si no es columna básica, guardar su índice
        if ceros != m-1 or unos != 1:
            NoBasics = NoBasics + [j]
    return NoBasics

In [12]:
def Vertex(Acol, b):
    # Número de ecuaciones
    m = b.nrows()
    # Número de variables
    n = len(Acol)

    # Buscar el vertice
    for i in [2..n-2]:
        for j in [i+1..n-1]:
            Asin = []
            # Sistema que queda al quitar las columnas i y j
            for k in [0..n-1]:
                if k != i and k != j:
                    Asin = Asin + [Acol[k]]
            # Matriz del sistema
            Anew = block_matrix(QQ, [[Asin[l] for l in [0..m-1]]])
            # Contador para comprobar la factibilidad
            M = 0
            # Si la matriz no es invertible no nos sirve
            if det(Anew) == 0:
                M = -1
            else:
                # Añadir la columna de los términos independientes
                Anewb = block_matrix([[Anew, b]])
                # Resolver el sistema
                Solve = Anewb.echelon_form()
                # Comprobar si la solución es factible
                for k in [0..m-1]:
                    if Solve[k][m] >= 0:
                        M += 1
            # Si todas las soluciones obtenidas son positivas
            if M == m:
                # Devolver el punto y los índices de las no básicas
                return [Solve[0][m], Solve[1][m], [i, j]]

In [13]:
def Simplex(A):
    # Número de filas sin contar la función objetivo
    m = len(A) - 1
    # Número de columnas sin contar términos independientes
    n = len(A[0]) - 1
    # Obtener las columnas del sistema
    Acol = Col(A)
    # Obtener los términos independientes
    b = Indep(A)
    # Obtener las variables no básicas del vértice inicial
    Nobasics = Vertex(Acol, b)[2]

    B = [[A[i][k] for k in [0..n]] for i in [0..m]]
    # Hacer básicos los pivotes iniciales
    # Le damos primero la "entrada" y despues la "salida"
    B = EntrSal(B, 0, Nobasics[0] - 1)
    B = EntrSal(B, 1, Nobasics[1] - 1)

    # Repetir mientras no se alcanza el óptimo
    while FuncObj(B) == False:
        # Decidir entrada
        e = 0
        Entrada = 0
        for k in [0..n-1]:
            if B[0][k] < e:
                e = B[0][k]
                Entrada = k
        # Decidir salida
        c = 0
        Salida = 0
        for k in [1..m]:
            if B[k][Entrada] > 0:
                if c == 0:
                    c = B[k][n] / B[k][Entrada]
                    Salida = k
                else:
                    if B[k][n] / B[k][Entrada] < c:
                        c = B[k][n] / B[k][Entrada]
                        Salida = k
        # Aplicar paso del algoritmo
        B = EntrSal(B, Entrada, Salida)
    # Obtener las variables no básicas finales
    Nobasicsb = EncNoBasics(B)
    # Obtener columnas de la tabla final
    Bcol = Col(B)
    # Obtener términos independientes
    bInd = Indep(B)
    # Lista de columnas básicas
    Bsin = []
    # Nos quedamos con las columnas que no son básicas
    for k in [0..n-1]:
        if k != Nobasicsb[0] and k != Nobasicsb[1]:
            Bsin += [Bcol[k]]
    # Crear la matriz del sistema final
    Bnew = block_matrix(QQ, [[Bsin[l] for l in [0..m-1]]])
    # Añadir la columna de los términos independientes
    Bnewb = block_matrix([[Bnew, bInd]])
    # Resolver el sistema
    Solve = Bnewb.echelon_form()
    # Devolver el vertice optimo y el valor de la funcion objetivo
    return [[Solve[0][m], Solve[1][m]], B[0][n]]

In [14]:
# Comprobar
P2 = [[-1, -3, 0, 0, 0, 0, 0], [1, 2, 1, 0, 0, 0, 12], [-4, 1, 0, 1, 0, 0, -3], [-1, -1, 0, 0, 1, 0, -2], [2, -1, 0, 0, 0, 1, 4]]
print("Solución:", Simplex(P2))

Solución: [[2, 5], 17]
